In [ ]:
import pandas as pd

matches_file = "matches.csv"
deliveries_file = "deliveries.csv"

matches_df = pd.read_csv(matches_file)
deliveries_df = pd.read_csv(deliveries_file)

matches_df.head(), deliveries_df.head()


(       id   season        city        date match_type player_of_match  \
 0  335982  2007/08   Bangalore  18-04-2008     League     BB McCullum   
 1  335983  2007/08  Chandigarh  19-04-2008     League      MEK Hussey   
 2  335984  2007/08       Delhi  19-04-2008     League     MF Maharoof   
 3  335985  2007/08      Mumbai  20-04-2008     League      MV Boucher   
 4  335986  2007/08     Kolkata  20-04-2008     League       DJ Hussey   
 
                                         venue                        team1  \
 0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
 1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
 2                            Feroz Shah Kotla             Delhi Daredevils   
 3                            Wankhede Stadium               Mumbai Indians   
 4                                Eden Gardens        Kolkata Knight Riders   
 
                          team2                  toss_winner toss_decision  \


In [ ]:
best_xgb_model = grid_search.best_estimator_

print("Best XGBoost model trained successfully!")


Best XGBoost model trained successfully!


In [ ]:
import joblib

joblib.dump(best_xgb_model, "final_xgb_model.pkl")

print("Best XGBoost model saved as 'final_xgb_model.pkl'!")


Best XGBoost model saved as 'final_xgb_model.pkl'!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_xgb = best_xgb_model.predict(test_df.drop(columns=["win"]))

print("\n===== XGBoost Model Evaluation =====")
print("Accuracy:", accuracy_score(test_df["win"], y_pred_xgb))
print("Classification Report:\n", classification_report(test_df["win"], y_pred_xgb))
print("Confusion Matrix:\n", confusion_matrix(test_df["win"], y_pred_xgb))



===== XGBoost Model Evaluation =====
Accuracy: 0.9974154041910215
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     12079
           1       1.00      1.00      1.00     13070

    accuracy                           1.00     25149
   macro avg       1.00      1.00      1.00     25149
weighted avg       1.00      1.00      1.00     25149

Confusion Matrix:
 [[12053    26]
 [   39 13031]]


In [ ]:
import pandas as pd
import numpy as np
import joblib

In [ ]:
user_input = {
    'inning': 1,
    'cum_runs': 80,
    'cum_wickets': 5,
    'overs_completed': 12.0,
    'target': 0,
    'batting_team': "Mumbai Indians",
    'bowling_team': "Chennai Super Kings",
    'venue': "Wankhede Stadium"
}

In [ ]:
if user_input['overs_completed'] > 0:
    current_run_rate = user_input['cum_runs'] / user_input['overs_completed']
else:
    current_run_rate = 0


In [ ]:
remaining_overs = 20 - user_input['overs_completed']
if user_input['inning'] == 2 and remaining_overs > 0:
    required_run_rate = (user_input['target'] - user_input['cum_runs']) / remaining_overs
else:
    required_run_rate = 0


In [ ]:
batting_encoder = joblib.load("batting_encoder.pkl")
bowling_encoder = joblib.load("bowling_encoder.pkl")
venue_encoder = joblib.load("venue_encoder.pkl")


In [ ]:
batting_team_encoded = batting_encoder.transform([user_input['batting_team']])[0]
bowling_team_encoded = bowling_encoder.transform([user_input['bowling_team']])[0]
venue_encoded = venue_encoder.transform([user_input['venue']])[0]

In [ ]:
input_data = {
    'inning': [user_input['inning']],
    'cum_runs': [user_input['cum_runs']],
    'cum_wickets': [user_input['cum_wickets']],
    'current_run_rate': [current_run_rate],
    'required_run_rate': [required_run_rate],
    'target': [user_input['target']],
    'batting_team_encoded': [batting_team_encoded],
    'bowling_team_encoded': [bowling_team_encoded],
    'venue_encoded': [venue_encoded]
}

input_df = pd.DataFrame(input_data)

In [ ]:
rf_model = joblib.load('final_rf_model.pkl')
xgb_model = joblib.load('final_xgb_model.pkl')
log_reg_model = joblib.load('final_logistic_model.pkl')


In [ ]:
rf_prediction = rf_model.predict(input_df)[0]
rf_prob = rf_model.predict_proba(input_df)[0]

xgb_prediction = xgb_model.predict(input_df)[0]
xgb_prob = xgb_model.predict_proba(input_df)[0]

log_reg_prediction = log_reg_model.predict(input_df)[0]
log_reg_prob = log_reg_model.predict_proba(input_df)[0]

In [ ]:
rf_winner = user_input['batting_team'] if rf_prediction == 1 else user_input['bowling_team']
xgb_winner = user_input['batting_team'] if xgb_prediction == 1 else user_input['bowling_team']
log_reg_winner = user_input['batting_team'] if log_reg_prediction == 1 else user_input['bowling_team']


In [ ]:
print("\n====== Prediction Results ======")
print(f"Random Forest: Predicted Winner → {rf_winner}, Probabilities (Loss, Win): {rf_prob}")
print(f"XGBoost: Predicted Winner → {xgb_winner}, Probabilities (Loss, Win): {xgb_prob}")
print(f"Logistic Regression: Predicted Winner → {log_reg_winner}, Probabilities (Loss, Win): {log_reg_prob}")


====== Prediction Results ======
Random Forest: Predicted Winner → Mumbai Indians, Probabilities (Loss, Win): [0.15 0.85]
XGBoost: Predicted Winner → Mumbai Indians, Probabilities (Loss, Win): [0.15 0.85]
Logistic Regression: Predicted Winner → Mumbai Indians, Probabilities (Loss, Win): [0.15 0.85]
